# Generator Comparison

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import math
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors
from datetime import datetime
from functools import partial
from os import path, makedirs

# Root paths
workspace_root = os.getcwd()
cafpyana_root = "/home/lpelegri/cafpyana"

# --- SYSTEM PATH CONFIGURATION ---
# Add workspace root
sys.path.insert(0, os.path.join(workspace_root, "../../"))
# Add main CAFpyana root
sys.path.insert(0, cafpyana_root)
sys.path.insert(0, os.path.join(workspace_root, "makedf"))
    
# --- EXTERNAL LIBRARIES ---
import lmfit
import uproot
from pandas.errors import PerformanceWarning

# --- REPO CLASSES & HELPERS ---
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh
from pyanalib.split_df_helpers import *
from pyanalib.covariance import *

# --- ANALYSIS VILLAGE IMPORTS ---
# Load high-level configs first
from analysis_village.cc1pi.var_configs import *
from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Load specific CC1Pi utilities
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

# Load Systematics (These rely on the paths set above)
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *

# --- SETTINGS ---
warnings.filterwarnings("ignore", category=PerformanceWarning)
np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
save_fig = True
show_plot = True

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/comparisons/"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, f"generator_comparison-{today_str}")

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Cross section model comparisons

In [ ]:
branches_nu = ["PDGnu", "cc", "Enu_true", "tgt", "ELep", "fScaleFactor", "RWWeight", "Mode",
                       "Q2", "q0", "q3", "x", "y",
                       "W_nuc_rest", "W", "W_genie", "flagCC0pi", 'Weight',
              "CosThetaAdler", "PhiAdler", "dalphat", "dpt", "dphit", "pnreco_C"]

branches_trk = ["pdg", "px", "py", "pz", "E"]

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/generator_samples/"

gibuu_filename = file_dir + "gibuu2025_patch5_cc_numu_gen1_flux.root"
events = uproot.open(gibuu_filename+":FlatTree_VARS")
gibuu_nus = events.arrays(branches_nu, library="pd")
gibuu_trks = events.arrays(branches_trk, library="pd")

genie_filename = file_dir + "14_1000180400_CC_v3_6_2_AR23_20i_02_000_SBND_gen1.flat.root"
events = uproot.open(genie_filename+":FlatTree_VARS")
genie_nus = events.arrays(branches_nu, library="pd")
genie_trks = events.arrays(branches_trk, library="pd")

neut_filename = file_dir + "neut.flat.root"
events = uproot.open(neut_filename+":FlatTree_VARS")
neut_nus = events.arrays(branches_nu, library="pd")
neut_trks = events.arrays(branches_trk, library="pd")

nuwro_filename = file_dir+ "bnb.sbnd.nuwro_26_00_00_gen1_flux.flat.root"
events = uproot.open(nuwro_filename+":FlatTree_VARS")
nuwro_nus = events.arrays(branches_nu, library="pd")
nuwro_trks = events.arrays(branches_trk, library="pd")


genie_ar25_filename = file_dir + "14_1000180400_CC_v3_6_2_AR25_20i_01_000_SBND_gen1.flat.root"
events = uproot.open(genie_ar25_filename+":FlatTree_VARS")
genie_ar25_nus = events.arrays(branches_nu, library="pd")
genie_ar25_trks = events.arrays(branches_trk, library="pd")

In [ ]:
print(neut_trks.pdg)
neut_trks_subset = neut_trks[:10000].copy()

In [ ]:
import awkward as ak
import numpy as np

import awkward as ak
import numpy as np

def get_trk_info(nudf, trkdf):
    # Loop setup
    pids = [13, 2212, 2112, 211, 111, 13, 211, 211, 2212]
    names = ["mu", "p","n", "pi", "pi0", "mu_P_100MeV_3000MeV", "pi_P_130MeV_2000MeV", "pi_P_85MeV_10000MeV", "p_P_325MeV_10000MeV"]
    lows = [0, 0, 0, 0, 0, 0.1, 0.13, 0.085, 0.325]
    highs = [1000, 1000, 1000, 1000, 1000, 3, 2, 1000, 1000]

    pdg = ak.Array(trkdf.pdg.values)
    abs_pdg = np.abs(pdg)
    mask = (abs_pdg != -1) & (abs_pdg != 22) & (abs_pdg < 1e8)
    
    print("Start arrays")
    pdg = pdg[mask]
    px = ak.Array(trkdf.px.values)[mask]
    py = ak.Array(trkdf.py.values)[mask]
    pz = ak.Array(trkdf.pz.values)[mask]
    print("Arrays done")
    
    p_mag_all = np.sqrt(px**2 + py**2 + pz**2)
    nudf["ntrks"] = ak.num(pdg, axis=1)

      
    for pid, pname, plow, phigh in zip(pids, names, lows, highs):
        print(f"Doing PID = {pid} ({pname})")
        
        # 1. Mask for PID
        pid_mask = (abs(pdg) == pid)
        thr_mask = pid_mask & (p_mag_all > plow) & (p_mag_all < phigh)
        
        # 2. Extract specific particle info
        p_filtered = p_mag_all[thr_mask]
        px_filtered = px[thr_mask]
        py_filtered = py[thr_mask]
        pz_filtered = pz[thr_mask]

        # 3. Find index of the MAXIMUM momentum for each event
        # keepdims=True allows us to use this index to pick from other arrays
        max_idx = ak.argmax(p_filtered, axis=1, keepdims=True)
        
        # 4. Use that index to get the "Leading" (highest P) track values
        # ak.firsts converts the single-element lists back to flat values/None
        leading_p = ak.firsts(p_filtered[max_idx])
        
        nudf[f"{pname}_p"] = ak.to_numpy(leading_p)
        nudf[f"{pname}_dirx"] = ak.to_numpy(ak.firsts(px_filtered[max_idx]) / leading_p)
        nudf[f"{pname}_diry"] = ak.to_numpy(ak.firsts(py_filtered[max_idx]) / leading_p)
        nudf[f"{pname}_dirz"] = ak.to_numpy(ak.firsts(pz_filtered[max_idx]) / leading_p)

        # 5. Count per PID within momentum window
        nudf[f"n{pname}"] = ak.to_numpy(ak.num(pdg[thr_mask], axis=1))

    print("Calculating TKI variables...")
    
    # 1. Hadronic Momentum Sum (Summing ALL protons and pions in the event)
    # We use axis=1 to sum across particles within each event
    p_mask = (abs(pdg) == 2212)
    pi_mask = (abs(pdg) == 211)
    
    p_had_x = ak.to_numpy(ak.sum(px[p_mask], axis=1) + ak.sum(px[pi_mask], axis=1))
    p_had_y = ak.to_numpy(ak.sum(py[p_mask], axis=1) + ak.sum(py[pi_mask], axis=1))
    
    # 2. Muon Transverse Momentum
    # Using the filtered muon components already stored in the dataframe
    p_mu_x = nudf["mu_P_100MeV_3000MeV_p"] * nudf["mu_P_100MeV_3000MeV_dirx"]
    p_mu_y = nudf["mu_P_100MeV_3000MeV_p"] * nudf["mu_P_100MeV_3000MeV_diry"]

    # --- cc1pi_deltapt ---
    # Magnitude of the transverse momentum imbalance vector
    d_px = p_mu_x + p_had_x
    d_py = p_mu_y + p_had_y
    nudf["cc1pi_dpt"] = np.sqrt(d_px**2 + d_py**2)

    # Magnitudes in the transverse plane
    mag_mu_t = np.sqrt(p_mu_x**2 + p_mu_y**2)
    mag_had_t = np.sqrt(p_had_x**2 + p_had_y**2)

    # --- cc1pi_deltaphit ---
    # Angle between transverse muon and transverse hadronic vectors
    dot_phi = -(p_mu_x * p_had_x + p_mu_y * p_had_y)
    norm_phi = mag_mu_t * mag_had_t
    
    safe_phi = (norm_phi > 0)
    nudf["cc1pi_dphit"] = np.full(len(nudf), -999.0)
    # acos(clip) prevents errors from float precision going slightly outside [-1, 1]
    nudf.loc[safe_phi, "cc1pi_dphit"] = np.arccos(np.clip(dot_phi[safe_phi] / norm_phi[safe_phi], -1.0, 1.0))

    # --- cc1pi_deltaalphat ---
    # Angle between transverse muon and the delta pT vector
    dot_alpha = -(p_mu_x * d_px + p_mu_y * d_py)
    norm_alpha = mag_mu_t * nudf["cc1pi_dpt"]
    
    safe_alpha = (norm_alpha > 0)
    nudf["cc1pi_dalphat"] = np.full(len(nudf), -999.0)
    nudf.loc[safe_alpha, "cc1pi_dalphat"] = np.arccos(np.clip(dot_alpha[safe_alpha] / norm_alpha[safe_alpha], -1.0, 1.0))

    print("Calculating mu-pi angle...")
    ux, uy, uz = nudf["mu_P_100MeV_3000MeV_dirx"], nudf["mu_P_100MeV_3000MeV_diry"], nudf["mu_P_100MeV_3000MeV_dirz"]
    px_dir, py_dir, pz_dir = nudf["pi_P_130MeV_2000MeV_dirx"], nudf["pi_P_130MeV_2000MeV_diry"], nudf["pi_P_130MeV_2000MeV_dirz"]
    
    dot_product = ux*px_dir + uy*py_dir + uz*pz_dir
    dot_product = np.clip(dot_product, -1.0, 1.0)
    nudf["mu_pi_angle"] = np.arccos(dot_product)
    
    return nudf

In [ ]:
genie_nus = get_trk_info(genie_nus, genie_trks)
genie_ar25_nus = get_trk_info(genie_ar25_nus, genie_ar25_trks)
#gibuu_nus = get_trk_info(gibuu_nus, gibuu_trks)
# genie_bugfix_nus = get_trk_info(genie_bugfix_nus, genie_bugfix_trks)
# genie_bugfix_lqcd_nus = get_trk_info(genie_bugfix_lqcd_nus, genie_bugfix_lqcd_trks)
# genie_bugfix_minerva_nus = get_trk_info(genie_bugfix_minerva_nus, genie_bugfix_minerva_trks)
#neut_nus = get_trk_info(neut_nus, neut_trks)
#nuwro_nus = get_trk_info(nuwro_nus, nuwro_trks)

In [ ]:
'''
import pandas as pd
import numpy as np
import awkward as ak

# 1. Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

for i, idx in enumerate(event_indices):
    print(f"\n{'='*100}")
    print(f"  ENTRY #{i} | Event Index: {idx}")
    print(f"{'='*100}")

    for p_type, col_names in particle_groups.items():
        # Identify NUS columns safely
        matched_nus_cols = [c for c in genie_small_nus.columns if (c[1] if isinstance(c, tuple) else c) in col_names]

        if not matched_nus_cols:
            continue

        print(f"\n>>> CATEGORY: {p_type}")
        
        # --- Calculate nprim for this event ---
        if idx in genie_trks.index:
            try:
                # Get raw PDGs for the specific event
                raw_pdgs = ak.to_numpy(ak.flatten(genie_trks.loc[[idx], find_multi_col(genie_trks, 'pdg')].values))
                abs_pdgs = np.abs(raw_pdgs)
                
                # Apply your specific restrictions:
                # Not 22 (photons) and < 1e8 (not heavy ions)
                # (np.abs != -1 is omitted here as it's redundant, but can be added if desired)
                prim_mask = (abs_pdgs != 22) & (abs_pdgs < 1e8)
                nprim = np.sum(prim_mask)
            except:
                nprim = "N/A"
        else:
            nprim = 0

        # --- Print NUS SUMMARY ---
        summary = genie_small_nus.loc[[idx], matched_nus_cols].copy()
        summary.columns = [c[1] if isinstance(c, tuple) else c for c in summary.columns]
        
        # Transpose and insert nprim at the top
        summary_t = summary.T
        # Rename the column for cleaner display
        summary_t.columns = ['Value']
        
        # Create a tiny dataframe for nprim to prepend it
        nprim_df = pd.DataFrame({'Value': [nprim]}, index=['nprim'])
        summary_display = pd.concat([nprim_df, summary_t])
        
        print(f"Summary Variables (with Filtered nprim):\n{summary_display}")

        # --- Calculate and Print TRKS DETAILS ---
        if idx in genie_trks.index:
            raw_trks = genie_trks.loc[[idx]]
            
            try:
                pdgs = ak.to_numpy(ak.flatten(raw_trks[find_multi_col(genie_trks, 'pdg')].values))
                px = ak.to_numpy(ak.flatten(raw_trks[find_multi_col(genie_trks, 'px')].values))
                py = ak.to_numpy(ak.flatten(raw_trks[find_multi_col(genie_trks, 'py')].values))
                pz = ak.to_numpy(ak.flatten(raw_trks[find_multi_col(genie_trks, 'pz')].values))
                energies = ak.to_numpy(ak.flatten(raw_trks[find_multi_col(genie_trks, 'E')].values))
                
                p_mag_calc = np.sqrt(px**2 + py**2 + pz**2)
                
                display_df = pd.DataFrame({
                    'pdg': pdgs,
                    'p_mag_calc': p_mag_calc,
                    'E': energies
                })
                
                print(f"\nIndividual Tracks (Calculated):\n{display_df}")
                
            except Exception as e:
                print(f"Calculation failed: {e}")
        else:
            print("Individual Tracks: (None found)")
    
    if i >= 1000: 
        break
'''

In [ ]:
# choose a variable to unfold, defined in variable_configs.py
#var_config = VariableConfig.all_evts()
#var_branch = "mu_p"

var_config = VariableConfig.muon_momentum()
var_branch = "mu_p"
#var_config = VariableConfig.pion_momentum()
#var_branch = "pi_P_130MeV_800MeV_p"
#var_config = VariableConfig.muon_direction()
#var_branch = "mu_dirz"
#var_config = VariableConfig.pion_direction()
#var_branch = "pi_dirz"
#var_config = VariableConfig.delta_pt()
#var_branch = "cc1pi_dpt"
#var_branch = "dpt"
# var_config = VariableConfig.delta_alpha_T()
# var_branch = "dalphat"
#var_branch = "cc1pi_dalphat"
# var_config = VariableConfig.delta_phi_T()
#var_branch = "cc1pi_dphit"
# var_branch = "dphit"
#var_config = VariableConfig.angle_between_candidates()
#var_branch = "mu_pi_angle"

In [ ]:
def get_signal(df):
    numuCC = (abs(df.PDGnu) == 14) & (df.cc == 1)
    topology = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_2000MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.ntrks - df.nmu - df.npi - df.np - df.nn == 0
    is_theta = df.mu_pi_angle < CTE.max_angle_between_candidates
    
    is_mu_p =  df.mu_p < 1
    return numuCC & topology & is_NpiNmuNnNp & is_theta & is_mu_p


In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/unfold_results"
unfold = np.load(file_dir + "/unfolding_result_" + var_config.var_save_name + ".npz")

In [ ]:
import numpy as np
from scipy.stats import chi2 as chi2_dist

def get_chi2(data, model, cov, n_params=0):
    delta = model - data 
    inv_cov = np.linalg.inv(cov)

    chi2 = delta @ inv_cov @ delta
    ndof = len(data) - n_params
    pval = chi2_dist.sf(chi2, ndof)

    return chi2, pval

In [ ]:
data_tot_pot = 4.4343664e+18
data_gates = 948132
integrated_fv_flux = 0.00015193910393930386/1e4

def get_xsec_unit():
    print("data_tot_pot: %.3e" %(data_tot_pot))

    integrated_flux = data_tot_pot * integrated_fv_flux
    print("Integrated flux: %.3e" % integrated_flux)

    
    V_SBND = (
        185 * 380 * 440 + 
        185 * 380 * 240 + 
        185 * 290 * 200
    )# cm3, the active volume of the detector 
    NTARGETS = RHO * V_SBND * N_A / M_AR
    
    print("# of targets: ", NTARGETS)

    
    flux_64 = np.float64(integrated_flux)
    targets_64 = np.float64(NTARGETS)
    
    denom = flux_64 * targets_64
    
    xsec_unit = 1. / denom
    # TODO: fix scalar overflow error in python v3.10+
    if xsec_unit == 0:
        print("XSEC_UNIT is 0, setting to 1e-38")
        xsec_unit = 1e-38
        
    print("xsec unit: ", xsec_unit)
    return xsec_unit
    
XSEC_UNIT = get_xsec_unit()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 0. Initialization & Config ---
nevts_dict = {}
do_clip = True 
plot_xsec = True 

# Binning setup
bins = var_config.bins
bin_centers = var_config.bin_centers
bin_widths = bins[1:] - bins[:-1]

# --- 1. Prepare Data and Covariances ---
unfolded_val = unfold['unfold']
unfold_cov_syst = unfold['SystUnfoldCov']
unfold_cov_stat = unfold['StatUnfoldCov']
if plot_xsec:
    unfolded_val = unfolded_val * XSEC_UNIT
    unfold_cov_syst = unfold_cov_syst * (XSEC_UNIT**2)
    unfold_cov_stat = unfold_cov_stat * (XSEC_UNIT**2)

# --- 2. Normalization / Shape Decomposition ---
# Use GENIE as the reference for decomposition
ref_gen = genie_nus[get_signal(genie_nus)]
n_ref_truth, _ = np.histogram(
    np.clip(ref_gen[var_branch], bins[0]+1e-5, bins[-1]-1e-5), 
    bins=bins, weights=40*ref_gen.fScaleFactor*ref_gen.Weight
)
if plot_xsec: n_ref_truth *= XSEC_UNIT


# Smear reference to match Reco-space Covariance
ref_model_smeared = unfold['AddSmear'] @ n_ref_truth

# Decompose Syst Covariance
SystUnfoldCov_norm, SystUnfoldCov_shape = Matrix_Decomp(ref_model_smeared, unfold_cov_syst)

# De-normalizing to find individual error components
unfold_uncert_norm = np.sqrt(np.abs(np.diag(SystUnfoldCov_norm)))
unfold_uncert_shape = np.sqrt(np.abs(np.diag(SystUnfoldCov_shape)))
unfold_uncert_stat = np.sqrt(np.abs(np.diag(unfold_cov_stat)))

# Combine Stat + Shape for the data points; Norm stays as bars at bottom
total_point_uncert = np.sqrt(unfold_uncert_stat**2 + unfold_uncert_shape**2)

# Normalize for differential plot
unfolded_per_width = unfolded_val / bin_widths
point_uncert_per_width = total_point_uncert / bin_widths
norm_uncert_per_width = unfold_uncert_norm / bin_widths

# --- 3. Initialize Figure ---
fig, ax = plt.subplots(figsize=(8.5, 7))

# Plot Norm Uncertainty (Gray bars)
norm_handle = ax.bar(bin_centers, norm_uncert_per_width, width=bin_widths, 
                     label='Syst. error (norm)', alpha=0.3, color='gray', zorder=1)

# Plot Unfolded Data Points
data_handle = ax.errorbar(bin_centers, unfolded_per_width, yerr=point_uncert_per_width, 
                          xerr=bin_widths/2, fmt='o', color='black', label='Unfolded Data',
                          capsize=2, elinewidth=1.5, markersize=5, zorder=10)

# --- 4. Generator Comparison Loop ---
generators = [genie_nus, genie_ar25_nus, gibuu_nus, nuwro_nus, neut_nus]
labels = ["GENIE AR23", "GENIE AR25", "GiBUU", "NuWro", "NEUT"]

generators = [genie_nus, genie_ar25_nus]
labels = ["GENIE AR23", "GENIE AR25"]

# --- 4. Generator Comparison Loop ---
for gendf, genname in zip(generators, labels):
    signal = gendf[get_signal(gendf)].copy()
    var = signal[var_branch]
    
    if do_clip:
        var = np.clip(var, bins[0] + 1e-5, bins[-1] - 1e-5)

    scale = 500 if genname == "GiBUU" else 1
    # Note: Using the average weight or the logic provided for the error scaling
    weights_val = (40 * signal.fScaleFactor * signal.Weight / scale)
    # Calculate truth -> forward fold to reco
    
    n_truth, _ = np.histogram(var, bins=bins, weights=weights_val)
    n_smeared = unfold['AddSmear'] @ n_truth
    
    # Unpack the result: counts go to n_truth_no_weight, edges are ignored (_)
    n_truth_no_weight, _ = np.histogram(var, bins=bins) 
    n_smeared_no_weight = unfold['AddSmear'] @ n_truth_no_weight
    # Calculate error based on your provided formula
    # We use np.where or a small epsilon to avoid division by zero if n_smeared is 0
    # Formula: sqrt(n_smeared / (XSEC_UNIT)) * XSEC_UNIT
    # (Simplified from your version as the weights are already inside n_smeared)
    with np.errstate(divide='ignore', invalid='ignore'):
        # Your formula: sqrt(counts/unit) * unit * weighted/unweighted
        # This accounts for the 'effective' weight of the smeared bin
        ratio = np.where(n_smeared_no_weight > 0, n_smeared / n_smeared_no_weight, 0)
        
        n_smeared_err = np.sqrt(np.abs(n_smeared_no_weight )) * ratio
        
    # Chi2 against total covariance
    total_cov = unfold_cov_syst
    chi2_val, _ = get_chi2(unfolded_val, n_smeared, total_cov)
    
    # Normalize for plotting
    n_plot_smeared = n_smeared / bin_widths
    n_plot_err = n_smeared_err / bin_widths
    
    # Plot the line
    line, = ax.step(bins, np.append(n_plot_smeared, n_plot_smeared[-1]), 
                    where='post', 
                    label=f"{genname} ($\chi^2$/ndf={chi2_val:.1f}/{len(n_smeared)})", 
                    linewidth=1.5)
    
    # Plot the error bars
    ax.errorbar(bin_centers, n_plot_smeared, yerr=n_plot_err, 
                fmt='none', color=line.get_color(), linewidth=1, capsize=0, alpha= 1)
    
    nevts_dict[genname] = n_smeared

# --- 5. Styling & Legend ---
max_y = max(unfolded_per_width.max(), max([(v/bin_widths).max() for v in nevts_dict.values()]))
ax.set_ylim(0, max_y * 1.55) 

ax.legend(
    loc='upper center', 
    bbox_to_anchor=(0.5, 1), # High enough for ncol=2
    ncol=2, 
    fontsize=11, 
    frameon=False,
    columnspacing=1.0
)

ax.set_xlabel(var_config.var_labels[0])
ax.set_ylabel(var_config.xsec_label)
ax.set_xlim(bins[0], bins[-1])

# Adjust for legend room
plt.subplots_adjust(top=0.82)

save_name = f"{save_fig_dir}/{var_config.var_save_name}-generator_comparison.pdf"
plt.savefig(save_name, bbox_inches="tight", dpi=300)
plt.show()


# --- 5. Ratio Plotting ---
fig_ratio, ax_ratio = plt.subplots(figsize=(9, 7))

# 1. Calculate Reference using Unfolded Data (per width)
# To compare counts-to-counts, we use the raw unfolded values (unfolded_val)
n_ref = unfolded_val

# Use the combined stat + shape uncertainty of the data as the reference error
n_ref_err = total_point_uncert

# Relative error of the unfolded data reference: sigma_data / data
with np.errstate(divide='ignore', invalid='ignore'):
    ref_rel_err = np.where(n_ref > 0, n_ref_err / n_ref, 0)

# 2. Plot the Grey Stat+Shape Error Band for the Unfolded Data centered at 1.0
ax_ratio.fill_between(bins, 
                      np.append(1 - ref_rel_err, 1 - ref_rel_err[-1]), 
                      np.append(1 + ref_rel_err, 1 + ref_rel_err[-1]), 
                      step='post', color='grey', alpha=0.3, zorder=1,
                      label='Unfolded Data Uncert. (Stat+Shape)')

# Horizontal line baseline at 1.0
ax_ratio.axhline(1.0, color='black', linestyle='--', linewidth=1.5, zorder=2)

# 3. Generator Loop for Ratio
for gendf, genname in zip(generators, labels):
    signal = gendf[get_signal(gendf)].copy()
    var_ratio = signal[var_branch]    
    if do_clip:
        var_ratio = np.clip(var_ratio, bins[0] + 1e-5, bins[-1] - 1e-5)

    scale = 500 if genname == "GiBUU" else 1
    weights_val = (40 * signal.fScaleFactor * signal.Weight / scale)
    
    # Calculate truth -> forward fold to reco
    n_model_truth, _ = np.histogram(var_ratio, bins=bins, weights=weights_val)
    n_model = unfold['AddSmear'] @ n_model_truth
    
    # Calculate model errors using weights squared -> forward fold
    n_model_w2, _ = np.histogram(var_ratio, bins=bins, weights=weights_val**2)
    n_model_err = np.sqrt(unfold['AddSmear'] @ n_model_w2)
   
    with np.errstate(divide='ignore', invalid='ignore'):
        # Ratio: Unfolded Data / Model
        ratio = np.where(n_model > 0, n_ref / n_model, 0)
        
        # Propagate Model-only relative uncertainty to the ratio bars
        gen_rel_err = np.where(n_model > 0, n_model_err / n_model, 0)
        ratio_err = ratio * gen_rel_err
    
    # Plot Generator Ratio Line
    line = ax_ratio.stairs(ratio, bins, label=f"Unfolded Data / {genname}", linewidth=1.5, zorder=3)
    
    # Plot Generator Error Bars
    ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_err, 
                      fmt='none', color=line.get_edgecolor(), linewidth=1, capsize=0, zorder=4)
    
# Ratio Aesthetics
var_unit_str = f" [{var_config.var_unit}]" if var_config.var_unit else ""
ax_ratio.set_xlabel(var_config.var_labels[0] + var_unit_str, fontsize=12)
ax_ratio.set_ylabel("Ratio (Unfolded Data / Model)", fontsize=12)
ax_ratio.set_xlim(bins[0], bins[-1])
ax_ratio.set_ylim(0.2, 1.8) 
ax_ratio.grid(True, linestyle=':', alpha=0.6)
ax_ratio.legend(frameon=False, ncol=2, loc='upper center')

plt.show()

In [ ]:
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100, filter_df = False )
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
# BNB MC
pot_weight_col = ('slc', 'wgt', '')

data_tot_pot = 5.947e+18
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_nu_df))

In [ ]:
new_columns = []
for c in mc_bnb_nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
mc_bnb_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

In [ ]:
def TruthInAV(data):
    xmin = -200
    xmax = 200 
    ymin = -200 
    ymax = 200 
    zmin = 0
    zmax = 500 
    
    pass_fv = (data.x > xmin) & (data.x < xmax) & (data.y > ymin) & (data.y < ymax) & (data.z < zmax) & (data.z > zmin)
    return pass_fv

In [ ]:
def TruthInFV(data):
    xmin = -200 + CTE.min_distance_to_wall_x_y
    xmax = 200 - CTE.min_distance_to_wall_x_y
    ymin = -200 + CTE.min_distance_to_wall_x_y
    ymax = 200 - CTE.min_distance_to_wall_x_y
    zmin = CTE.min_distance_to_first_z_wall
    zmax = 500 - CTE.min_distance_to_last_z_wall
    
    pass_fv = (data.x > xmin) & (data.x < xmax) & (data.y > ymin) & (data.y < ymax) & (data.z < zmax) & (data.z > zmin)
    return pass_fv

In [ ]:
def get_signal(df):
    numuCC = (abs(df.PDGnu) == 14) & (df.cc == 1)
    topology = (df.nmu_P_100MeV_3000MeV == 1) & (df.npi_P_130MeV_2000MeV == 1) & (df.npi_P_85MeV_10000MeV == 1)
    is_NpiNmuNnNp = df.ntrks - df.nmu - df.npi - df.np - df.nn == 0
    is_theta = df.mu_pi_angle < CTE.max_angle_between_candidates
    
    is_mu_p =  df.mu_p < 1
    
    return numuCC 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

generators = [genie_nus, genie_ar25_nus]
labels = ["GENIE AR23", "GENIE AR25"]

# --- 1. Setup Bins and Data ---
#var_config = VariableConfig.muon_momentum()
#var_branch = "mu_p"

var_config = VariableConfig.delta_phi_T()
var_branch = "cc1pi_dphit"

var_column = var_config.var_nu_col

bins = var_config.bins
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_widths = np.diff(bins)
wgt_col = ('truth', 'slc', 'wgt', '') 

# Get Reference Truth
topology_mask = (mc_bnb_nu_df.truth.nmu_P_100MeV_3000MeV == 1) & (mc_bnb_nu_df.truth.npi_P_130MeV_800MeV == 1)  & (mc_bnb_nu_df.truth.npi_P_85MeV_10000MeV == 1)
is_NpiNmuNnNp = mc_bnb_nu_df.truth.nprim - mc_bnb_nu_df.truth.nmu - mc_bnb_nu_df.truth.npi - mc_bnb_nu_df.truth.np - mc_bnb_nu_df.truth.nn == 0
numuCC_mask = (mc_bnb_nu_df.truth.iscc == 1) & TruthInFV(mc_bnb_nu_df.truth.position)  & (abs(mc_bnb_nu_df.truth.pdg) == 14)
is_theta = mc_bnb_nu_df.truth.true_var.true_mu_pi_angle <  CTE.max_angle_between_candidates

#signal_df = mc_bnb_nu_df[numuCC_mask & topology_mask & is_NpiNmuNnNp & is_theta].copy()
signal_df = mc_bnb_nu_df[numuCC_mask]
#signal_df = mc_bnb_nu_df[mc_bnb_nu_df.truth.nu_categ == "CC1pi"].copy()

# --- CLIPPING REFERENCE TRUTH ---
var_truth = signal_df[var_column]
# We clip values to be slightly inside the first/last bin edges
var_truth_clipped = np.clip(var_truth, bins[0] + 1e-5, bins[-1] - 1e-5)
w_truth = signal_df[wgt_col]

# --- 2. Calculate Reference Histograms ---
n_truth_evts, _ = np.histogram(var_truth_clipped, bins=bins, weights=w_truth)
n_truth_w2, _ = np.histogram(var_truth_clipped, bins=bins, weights=w_truth**2)
n_truth_err = np.sqrt(n_truth_w2)

# Scale and Normalize
n_plot_truth = (n_truth_evts * XSEC_UNIT) / bin_widths
n_plot_truth_err = (n_truth_err * XSEC_UNIT) / bin_widths

# --- 3. Main Plotting ---
fig, ax = plt.subplots(figsize=(9, 7))

ax.stairs(n_plot_truth, bins, color='black', linewidth=2.5, 
          label=r'Reference MC Truth ($CC1\pi$)', zorder=10)
ax.errorbar(bin_centers, n_plot_truth, yerr=n_plot_truth_err, 
            fmt='none', color='black', linewidth=1.5, capsize=2)

# --- 4. Generator Loop ---
for gendf, genname in zip(generators, labels):
    sig_mask = get_signal(gendf)
    signal = gendf[sig_mask].copy()
    
    # --- CLIPPING GENERATOR DATA ---
    var_gen = signal[var_branch]
    var_gen_clipped = np.clip(var_gen, bins[0] + 1e-5, bins[-1] - 1e-5)
    
    scale = 500 if genname == "GiBUU" else 1
    weights_val = (40 * signal.fScaleFactor * signal.Weight / scale)
    
    n_gen, _ = np.histogram(var_gen_clipped, bins=bins, weights=weights_val)
    n_gen_w2, _ = np.histogram(var_gen_clipped, bins=bins, weights=weights_val**2)
    
    n_plot_gen = n_gen / bin_widths
    ax.stairs(n_plot_gen, bins, linewidth=1.5, alpha=0.8, label=genname)

# Aesthetics
ax.set_xlabel(var_config.var_plot_name + " " + var_config.var_unit , fontsize=12)
ax.set_ylabel(var_config.xsec_label, fontsize=12)
ax.set_xlim(bins[0], bins[-1])
ax.set_ylim(0, np.max(n_plot_truth) * 1.5)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=False, ncol=2)
plt.show()

# --- 5. Ratio Plotting ---
fig_ratio, ax_ratio = plt.subplots(figsize=(9, 7))

# 1. Calculate Reference Histogram and its relative error
n_ref, _ = np.histogram(var_truth_clipped, bins=bins, weights=w_truth * XSEC_UNIT)
n_ref_w2, _ = np.histogram(var_truth_clipped, bins=bins, weights=(w_truth * XSEC_UNIT)**2)

# Relative error of the reference: sigma_ref / n_ref
with np.errstate(divide='ignore', invalid='ignore'):
    ref_rel_err = np.where(n_ref > 0, np.sqrt(n_ref_w2) / n_ref, 0)

# 2. Plot the Grey Stat Error Bar for Reference
# We use 'step' mode for fill_between to match the histogram bins
ax_ratio.fill_between(bins, 
                      np.append(1 - ref_rel_err, 1 - ref_rel_err[-1]), 
                      np.append(1 + ref_rel_err, 1 + ref_rel_err[-1]), 
                      step='post', color='grey', alpha=0.3, zorder=1,
                      label='Ref. MC Stat. Uncert.')

# Horizontal line at 1.0
ax_ratio.axhline(1.0, color='black', linestyle='--', linewidth=1.5, zorder=2)

# 3. Generator Loop for Ratio
for gendf, genname in zip(generators, labels):
    signal = gendf[get_signal(gendf)].copy()
    var_ratio = np.clip(signal[var_branch], bins[0] + 1e-5, bins[-1] - 1e-5)
    
    scale = 500 if genname == "GiBUU" else 1
    weights_val = (40 * signal.fScaleFactor * signal.Weight / scale)
    
    n_model, _ = np.histogram(var_ratio, bins=bins, weights=weights_val)
    n_model_w2, _ = np.histogram(var_ratio, bins=bins, weights=weights_val**2)
   
    with np.errstate(divide='ignore', invalid='ignore'):
        # Ratio: Ref / Model
        ratio = np.where(n_model > 0, n_ref / n_model, 0)
        
        # Propagate Generator-only error for the error bars
        # sigma_ratio = ratio * (sigma_model / n_model)
        gen_rel_err = np.where(n_model > 0, np.sqrt(n_model_w2) / n_model, 0)
        ratio_err = ratio * gen_rel_err
    
    # Plot Generator Ratio
    line = ax_ratio.stairs(ratio, bins, label=f"Ref / {genname}", linewidth=1.5, zorder=3)
    
    # Plot Generator Error Bars
    ax_ratio.errorbar(bin_centers, ratio, yerr=ratio_err, 
                      fmt='none', color=line.get_edgecolor(), linewidth=1, capsize=0, zorder=4)
    
# Ratio Aesthetics
ax_ratio.set_xlabel(var_config.var_plot_name + " " + var_config.var_unit, fontsize=12)
ax_ratio.set_ylabel("Ratio (Reference / Model)", fontsize=12)
ax_ratio.set_xlim(bins[0], bins[-1])
ax_ratio.set_ylim(0.2, 1.8) 
ax_ratio.grid(True, linestyle=':', alpha=0.6)
ax_ratio.legend(frameon=False, ncol=2, loc='upper center')

plt.show()

In [ ]:
data_tot_pot = 5.947e+18
data_gates = 948132
integrated_fv_flux = 0.00014939363669001503/1e4

def get_xsec_unit():
    print("data_tot_pot: %.3e" %(data_tot_pot))

    integrated_flux = data_tot_pot * integrated_fv_flux
    print("Integrated flux: %.3e" % integrated_flux)

    V_SBND = (
        400*400*500
    )# cm3, the active volume of the detector 
    NTARGETS = RHO * V_SBND * N_A / M_AR
    
    print("# of targets: ", NTARGETS)
    
    flux_64 = np.float64(integrated_flux)
    targets_64 = np.float64(NTARGETS)

    
    denom = flux_64 * targets_64
    
    xsec_unit = 1. / denom
    # TODO: fix scalar overflow error in python v3.10+
    if xsec_unit == 0:
        print("XSEC_UNIT is 0, setting to 1e-38")
        xsec_unit = 1e-38
        
    print("xsec unit: ", xsec_unit)
    return xsec_unit
    
XSEC_UNIT = get_xsec_unit()